# Slang Generation Demo

This demo will illustrate the basic workflow of the slang generation code accompanying the TACL paper *[A Computational Framework for Slang Generation](https://direct.mit.edu/tacl/article/doi/10.1162/tacl_a_00378/100687/A-Computational-Framework-for-Slang-Generation)* using slang definition data from Urban Dictionary (UD) and conventional definition data from WordNet.

To run this tutorial, you will need the following dependencies:

- Python 3
- Numpy
- Scipy
- tqdm
- NLTK
- Gensim
- PyTorch (torch)
- SBERT (sentence_transformers)
- [CatGO](https://github.com/zhewei-sun/CatGO)


In [1]:
import numpy as np
import torch
import shutil
import copy

We first create a symbolic link pointing to the library. You will need to change the destination if your code sits in a different directory. 

In [2]:
! ln -s ../Code slanggen

ln: failed to create symbolic link 'slanggen/Code': File exists


CatGO is a library that optimizes and runs models of categorization and can be obtained [here](https://github.com/zhewei-sun/CatGO). Once you have downloaded the code, please link it by replacing the target directory of the simlink below. 

In [3]:
! ln -s ../../CatGO CatGO
import nltk
nltk.download('stopwords')

ln: failed to create symbolic link 'CatGO/CatGO': File exists


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/kyx8046/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
from slanggen.util import *
from slanggen.dataloader import WN_Dataset, Urban_Dataset, OSD_Dataset, ZH_Dataset
from slanggen.encoder import FTEncoder, FTCachedEncoder, SBertEncoder, SenseEncoder, dump_vanilla_embeddings
from slanggen.contrastive import SlangGenTrainer
from slanggen.model import SlangGenModel

/projects/b1170/users/kyx8046/miniconda3/envs/slanggen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Specify a PyTorch device if necessary:

In [5]:
import os, torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("is_available =", torch.cuda.is_available())
print("device_count =", torch.cuda.device_count())

torch.cuda.set_device(0)

CUDA_VISIBLE_DEVICES = 0
is_available = True
device_count = 1


Load conventional definition data using the builtin dataloaders. The *.npy* file loaded below contains a pre-processed version of WordNet definition sentences for all words that appear in both WordNet and UD.

In [6]:
mix_conv_data = WN_Dataset('mix_multilingual_conv_data_all_OD_chime.npy')

print(mix_conv_data)

# EN conv dataset, used for prediction test
en_conv_data = WN_Dataset('wordnet_urban.npy')
print(en_conv_data)

# ZH conv dataset, used for prediction test
zh_conv_data = WN_Dataset('ZH_conv_data_chime.npy')
print(zh_conv_data)

# RU conv dataset, used for prediction test
ru_conv_data = WN_Dataset('RU_conv_data.npy')
print(ru_conv_data)

Dataset Name: 
Total Definition Entries: 12614
Vocab Size: 3494

Dataset Name: 
Total Definition Entries: 9759
Vocab Size: 1464

Dataset Name: 
Total Definition Entries: 692
Vocab Size: 692

Dataset Name: 
Total Definition Entries: 961
Vocab Size: 961



Load slang definition data. The *.npy* file loaded below is a pre-processed version of the data released in this repository.

In [7]:
mix_slang_data = OSD_Dataset('mix_multilingual_slang_data_all_OD_chime.npy', mix_conv_data)

print(mix_slang_data)

# EN/ZH slang dataset, used for prediction test
en_slang_data = OSD_Dataset('OSD_data.npy', en_conv_data)
zh_slang_data = ZH_Dataset('ZH_slang_data_chime.npy', zh_conv_data)
ru_slang_data = ZH_Dataset('RU_slang_data.npy', ru_conv_data)

Dataset Name: 
Total Definition Entries: 4688
Vocab Size: 3288



If you wish to use your own dataset, please create a dataloader object inheriting either *ConvDataset* or *SlangDataset* abstract classes found in *dataloader.py* and following the example data specifications in *dataloader.WN_Dataset* and *dataloader.Urban_Dataset*.

Now let's create a directory to store our results and load in some pre-generated data indices for train-test split:

In [8]:
shutil.rmtree('Results')  
create_directory('Results')

In [9]:
out_dir='Results/'

dataset_mix = mix_slang_data
slang_inds_mix = DataIndex(np.load('train_ind_mix_multilingual_all_OD_chime.npy'), np.load('dev_ind_mix_multilingual_all_OD_chime.npy'), np.load('test_ind_mix_multilingual_all_OD_chime.npy'))

# For prediction tests
dataset_mix_en = en_slang_data
slang_inds_en = DataIndex(np.load('train_ind_osd.npy'), np.load('dev_ind_osd.npy'), np.load('test_ind_osd.npy'))

dataset_mix_zh = zh_slang_data
slang_inds_zh = DataIndex(np.load('train_ind_ZH_chime.npy'), np.load('dev_ind_ZH_chime.npy'), np.load('test_ind_ZH_chime.npy'))

dataset_mix_ru = ru_slang_data
slang_inds_ru = DataIndex(np.load('train_ind_ru.npy'), np.load('dev_ind_ru.npy'), np.load('test_ind_ru.npy'))

# #Test the performance on other languages
# dataset_zh_zh = zh_data
# slang_inds_zh_zh = DataIndex(np.load('train_ind_zh_zh.npy'), np.load('dev_ind_zh_zh.npy'), np.load('test_ind_zh_zh.npy'))

# dataset_ru_ru = ru_data
# slang_inds_ru_ru = DataIndex(np.load('train_ind_ru_ru.npy'), np.load('dev_ind_ru_ru.npy'), np.load('test_ind_ru_ru.npy'))

The following encoder objects initializes a fastText encoder used for collaborative filtering. *FTEncoder* can be used to read in the original fastText embedding file. For efficiency, we have cached the words we need and use a cached encoder instead.

ft_encoder = FTEncoder('path to crawl-300d-2M-subword.vec')

In [10]:
ft_encoder = FTCachedEncoder('ft_embed_cache_Urban.pickle')

The following commands sets up the contrastive trainer and the slang generation model:

In [11]:
trainer_mix = SlangGenTrainer(dataset_mix, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)

Encoding vocab for word_dist with paraphrase-multilingual-mpnet-base-v2 ...


Batches: 100%|█████████████████████████████████████████████████████████████████| 26/26 [00:02<00:00, 10.18it/s]


You can modify the 'embed_name' param below to choose the sense encoding model, here is a list of supported models:
- bert-base-nli-mean-tokens -> 'SBERT_contrastive' (default)
- sentence-t5-base -> 'SBERT_t5'
- paraphrase-multilingual-MiniLM-L12-v2 -> 'SBERT-multilingual-MiniLM-L12-v2'
- LaBSE -> 'SBERT_LaBSE'
- paraphrase-multilingual-mpnet-base-v2 -> 'SBERT_mpnet'
- intfloat/multilingual-e5-base -> 'SBERT_e5_base'
- intfloat/multilingual-e5-large -> 'SBERT_e5_large'

In [12]:
model_mix = SlangGenModel(trainer_mix, data_dir=out_dir, embed_name='SBERT_mpnet')

params = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'cf_prototype_5', 'prior':None, 'prior_name':'uniform', 'contr_params':None}

Invoke *model.train_contrastive* to train the contrastively learned sense embedding model:

Note: you can set mode to 'head' to train only the triplet head, or 'whole' to train both the sense encoding and the head. Also it's optional to change the fold_name if using a new dataset.

In [13]:
model_mix.train_contrastive(slang_inds_mix, fold_name='mix_all_mpnet', params=params, mode='head')

Generating contrative pairs...


100%|████████████████████████████████████████████████████████████████████████| 234/234 [00:31<00:00,  7.32it/s]


Complete!
Training contrastive model with head
Generating triplet data for contrastive training...
Sampled 64552 Triplets
Sampled 4359 Triplets
Complete!


Epoch 1/4 train: 100%|█████████████████████████████████████████████████████| 8022/8022 [03:45<00:00, 35.54it/s]


Epoch 1 average training loss: 0.5303


...

Epoch 3/4  train_loss=0.3400  val_loss=0.5051  (saved best val=0.5051)


Epoch 4/4 train: 100%|█| 8022/8022 [03:46<00:00, 35.47it/s]


Epoch 4 average training loss: 0.3074


Epoch 4/4 val: 100%|██████████████████████████████████████████████████████████████| 543/543 [00:14<00:00, 37.89it/s]


Epoch 4/4  train_loss=0.3074  val_loss=0.4942  (saved best val=0.4942)
Done. best_val_loss=0.4942
Cleared GPU cache before loading model
Encoding sense definitions...
Complete!


In [14]:
# Now we want to see how the mixed dataset performs on each language

# Step 1: Create symbolic links
!mkdir -p Results/mix_all_mpnet_en/SBERT_data
!mkdir -p Results/mix_all_mpnet_ru/SBERT_data
!mkdir -p Results/mix_all_mpnet_zh/SBERT_data

!cp Results/mix_all_mpnet/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_all_mpnet_en/SBERT_data/
!cp Results/mix_all_mpnet/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_all_mpnet_ru/SBERT_data/
!cp Results/mix_all_mpnet/SBERT_data/SBERT_mpnet_with_head.pt Results/mix_all_mpnet_zh/SBERT_data/

params['embed_name'] = 'SBERT_mpnet' 

trainer_mix_en = SlangGenTrainer(dataset_mix_en, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_mix_en   = SlangGenModel(trainer_mix_en, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_mix_ru = SlangGenTrainer(dataset_mix_ru, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_mix_ru   = SlangGenModel(trainer_mix_ru, data_dir=out_dir, embed_name='SBERT_mpnet')

trainer_mix_zh = SlangGenTrainer(dataset_mix_zh, word_encoder=ft_encoder, out_dir=out_dir, verbose=True)
model_mix_zh   = SlangGenModel(trainer_mix_zh, data_dir=out_dir, embed_name='SBERT_mpnet')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Encoding vocab for word_dist with paraphrase-multilingual-mpnet-base-v2 ...


Batches: 100%|████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 68.06it/s]


Encoding vocab for word_dist with paraphrase-multilingual-mpnet-base-v2 ...


Batches: 100%|████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 41.54it/s]


Encoding vocab for word_dist with paraphrase-multilingual-mpnet-base-v2 ...


Batches: 100%|████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 58.61it/s]


In [15]:
trainer_mix_en.get_trained_embeddings(slang_inds_en, fold_name='mix_all_mpnet_en', model_path='SBERT_mpnet')
trainer_mix_ru.get_trained_embeddings(slang_inds_ru, fold_name='mix_all_mpnet_ru', model_path='SBERT_mpnet')
trainer_mix_zh.get_trained_embeddings(slang_inds_zh, fold_name='mix_all_mpnet_zh', model_path='SBERT_mpnet')

params = {'embed_name':'SBERT_mpnet', 'out_name':'predictions', 'model':'prototype', 'prior':None, 'prior_name':'uniform', 'contr_params':None}

model_mix_en.train_categorization(slang_inds_en, fold_name='mix_all_mpnet_en', params=params)
results_mix_en = model_mix_en.get_results(fold_name='mix_all_mpnet_en', mode='train', params=params)

model_mix_ru.train_categorization(slang_inds_ru, fold_name='mix_all_mpnet_ru', params=params)
results_mix_ru = model_mix_ru.get_results(fold_name='mix_all_mpnet_ru', mode='train', params=params)

model_mix_zh.train_categorization(slang_inds_zh, fold_name='mix_all_mpnet_zh', params=params)
results_mix_zh = model_mix_zh.get_results(fold_name='mix_all_mpnet_zh', mode='train', params=params)

Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Cleared GPU cache before loading model
Encoding sense definitions...
Complete!
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.86it/s]


Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  8.19it/s]


Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 10.72it/s]


In [16]:
print('Performance on train split - EN')
N_train_dev_mix_en = dataset_mix_en.N_total - slang_inds_en.test.shape[0]
train_rankings_mix_en = get_rankings(results_mix_en, np.arange(N_train_dev_mix_en), dataset_mix_en.vocab_ids[np.concatenate(slang_inds_en)])
np.mean(get_roc(train_rankings_mix_en, dataset_mix_en.V))

Performance on train split - EN


0.7520232802423295

In [17]:
print('Performance on train split - RU')
N_train_dev_mix_ru = dataset_mix_ru.N_total - slang_inds_ru.test.shape[0]
train_rankings_mix_ru = get_rankings(results_mix_ru, np.arange(N_train_dev_mix_ru), dataset_mix_ru.vocab_ids[np.concatenate(slang_inds_ru)])
np.mean(get_roc(train_rankings_mix_ru, dataset_mix_ru.V))

Performance on train split - RU


0.7609400301367398

In [18]:
print('Performance on train split - ZH')
N_train_dev_mix_zh = dataset_mix_zh.N_total - slang_inds_zh.test.shape[0]
train_rankings_mix_zh = get_rankings(results_mix_zh, np.arange(N_train_dev_mix_zh), dataset_mix_zh.vocab_ids[np.concatenate(slang_inds_zh)])
np.mean(get_roc(train_rankings_mix_zh, dataset_mix_zh.V))

Performance on train split - ZH


0.8217797819848565

In [19]:
model_mix_en.predict_testset(slang_inds_en, fold_name='mix_all_mpnet_en', params=params)
results_mix_en = model_mix_en.get_results(fold_name='mix_all_mpnet_en', mode='test', params=params)

Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|█████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 99.92it/s]


In [20]:
model_mix_ru.predict_testset(slang_inds_ru, fold_name='mix_all_mpnet_ru', params=params)
results_mix_ru = model_mix_ru.get_results(fold_name='mix_all_mpnet_ru', mode='test', params=params)

Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 136.90it/s]


In [21]:
model_mix_zh.predict_testset(slang_inds_zh, fold_name='mix_all_mpnet_zh', params=params)
results_mix_zh = model_mix_zh.get_results(fold_name='mix_all_mpnet_zh', mode='test', params=params)

Params: {'embed_name': 'SBERT_mpnet', 'out_name': 'predictions', 'model': 'prototype', 'prior': None, 'prior_name': 'uniform', 'contr_params': None}
Pre-processing Distances...
Pre-processing Complete!
Optimizing Kernels...


100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 260.76it/s]


In [22]:
print('Performance on test split - EN')
inds = slang_inds_en.test                      
labels = dataset_mix_en.vocab_ids                 
test_rankings_mix_en = get_rankings(results_mix_en, inds, labels)
np.mean(get_roc(test_rankings_mix_en, dataset_mix_en.V))

Performance on test split - EN


0.7607073643410853

In [23]:
print('Performance on test split - RU')
inds = slang_inds_ru.test                      
labels = dataset_mix_ru.vocab_ids                 
test_rankings_mix_ru = get_rankings(results_mix_ru, inds, labels)
np.mean(get_roc(test_rankings_mix_ru, dataset_mix_ru.V))

Performance on test split - RU


0.755431023641851

In [24]:
print('Performance on test split - ZH')
inds = slang_inds_zh.test                      
labels = dataset_mix_zh.vocab_ids                 
test_rankings_mix_zh = get_rankings(results_mix_zh, inds, labels)
np.mean(get_roc(test_rankings_mix_zh, dataset_mix_zh.V))

Performance on test split - ZH


0.8174746743849494

In [25]:
import numpy as np
import pandas as pd
from slanggen.encoder import SBertWithHeadEncoder, SBertEncoder

# 1) paths
csv_path = "input_def_pairs_OD_chime.csv"

# trained checkpoint
ckpt_path = "Results/mix_all_mpnet/SBERT_data/SBERT_mpnet_with_head.pt"
base_model = "paraphrase-multilingual-mpnet-base-v2"

# vanilla model name
vanilla_model = "paraphrase-multilingual-mpnet-base-v2"

# 2) load data
df = pd.read_csv(csv_path)

# 3) choose columns
col_a = "conv_def_en"
col_b = "slang_definition_en"

texts_a = df[col_a].fillna("").astype(str).tolist()
texts_b = df[col_b].fillna("").astype(str).tolist()

# 4A) load trained encoder (SBERT + trained head)
enc_trained = SBertWithHeadEncoder(base_model, ckpt_path)
emb_a_tr = enc_trained.encode_sentences(texts_a, batch_size=64)  # already L2-normalized
emb_b_tr = enc_trained.encode_sentences(texts_b, batch_size=64)

# 4B) load vanilla SBERT encoder
enc_vanilla = SBertEncoder(sbert_model_name=vanilla_model)
emb_a_vn = enc_vanilla.encode_sentences(texts_a)  # already L2-normalized
emb_b_vn = enc_vanilla.encode_sentences(texts_b)

# 5A) trained pairwise metrics (row-wise)
tr_cos_sim = np.sum(emb_a_tr * emb_b_tr, axis=1)
tr_cos_dist = 1.0 - tr_cos_sim
tr_euclidean = np.linalg.norm(emb_a_tr - emb_b_tr, axis=1)
tr_manhattan = np.sum(np.abs(emb_a_tr - emb_b_tr), axis=1)

# 5B) vanilla pairwise metrics (row-wise)
vn_cos_sim = np.sum(emb_a_vn * emb_b_vn, axis=1)
vn_cos_dist = 1.0 - vn_cos_sim
vn_euclidean = np.linalg.norm(emb_a_vn - emb_b_vn, axis=1)
vn_manhattan = np.sum(np.abs(emb_a_vn - emb_b_vn), axis=1)

# 6) save
out = df.copy()

# vanilla
out["vanilla_cos_sim"] = vn_cos_sim
out["vanilla_cos_dist"] = vn_cos_dist
out["vanilla_euclidean"] = vn_euclidean
out["vanilla_manhattan"] = vn_manhattan

# trained
out["trained_cos_sim"] = tr_cos_sim
out["trained_cos_dist"] = tr_cos_dist
out["trained_euclidean"] = tr_euclidean
out["trained_manhattan"] = tr_manhattan

out_path = "def_pairs_dis_mpnet_OD_chime.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path}, rows={len(out)}")

Cleared GPU cache before loading model
Saved: def_pairs_dis_mpnet_OD_chime.csv, rows=3861
